# Limpieza de Raw Data de Ventas

Este notebook muestra un flujo didáctico de limpieza de datos tipo **Raw → Silver → Gold**.

## Objetivo

Leer el archivo `Practica_Limpieza_Raw_Data_Ventas.xlsx`, limpiar la hoja `Ventas_Raw`, separar registros válidos y rechazados, crear una bitácora y generar un nuevo archivo Excel de salida.

## Qué problemas se limpian

- Espacios al inicio/final y dobles espacios.
- Mayúsculas/minúsculas inconsistentes.
- Fechas en diferentes formatos.
- Números guardados como texto, moneda o formatos regionales.
- Descuentos en decimal, porcentaje o texto.
- Categorías, ciudades, productos, métodos de pago y estados fuera de catálogo.
- Duplicados.
- Cantidades inválidas.
- Precios inválidos.
- Totales reportados inconsistentes.
- Emails inválidos como advertencia.


In [6]:
# Si estás ejecutando esto en una computadora nueva, primero instala estas librerías:
# !pip install pandas openpyxl


In [7]:
import pandas as pd
import numpy as np
import re
import unicodedata
from datetime import datetime
from pathlib import Path


In [8]:
# ---------------------------------------------------------
# 1. Configuración de archivos
# ---------------------------------------------------------

input_file = Path("Practica_Limpieza_Raw_Data_Ventas.xlsx")

# Si el archivo está en otra carpeta, cambia la ruta:
# input_file = Path(r"C:\Users\TuUsuario\Downloads\Practica_Limpieza_Raw_Data_Ventas.xlsx")

print("Archivo de entrada:", input_file.resolve())


Archivo de entrada: C:\Users\david\Desktop\Unicah\BigData\Periodo-2-2026\S2\Kit_Limpieza_Raw_Data_Ventas\Practica_Limpieza_Raw_Data_Ventas.xlsx
Archivo de salida: C:\Users\david\Desktop\Unicah\BigData\Periodo-2-2026\S2\Kit_Limpieza_Raw_Data_Ventas\Salida_Limpieza_Ventas_Silver_Gold.xlsx


In [9]:
# ---------------------------------------------------------
# 2. Cargar raw data
# ---------------------------------------------------------

# keep_default_na=False ayuda a conservar textos como "N/A" para analizarlos como errores de origen.
df_raw = pd.read_excel(input_file, sheet_name="Ventas_Raw", keep_default_na=False)

print("Filas y columnas del raw data:", df_raw.shape)
display(df_raw.head(10))


Filas y columnas del raw data: (52, 17)


,IdVenta,FechaVenta,ClienteId,ClienteNombre,Ciudad,Producto,Categoria,Cantidad,PrecioUnitario,Descuento,MetodoPago,Estado,Vendedor,EmailCliente,Telefono,TotalReportado,Observaciones_Origen
0,V-1001,2026-05-01 00:00:00,C001,Ana López,Tegucigalpa,Laptop,Tecnología,1,14500,0.05,Tarjeta,Pagado,M. Reyes,ana.lopez@example.com,9988-7766,13775,
1,V-1001,2026-05-01 00:00:00,C001,Ana López,Tegucigalpa,Laptop,Tecnología,1,14500,0.05,Tarjeta,Pagado,M. Reyes,ana.lopez@example.com,9988-7766,13775,Posible duplicado exacto
2,V-1002,01/05/2026,c002,Carlos Mejía,San Pedro Sula,mouse,tecnologia,2,L. 350.00,0,efectivo,Pagado,A. Cruz,carlos.mejia@example.com,9999 1122,700,
3,V-1003,2026/05/02,C003,María Santos,SPS,Teclado,tecnologia,1,"450,50",5%,Transferencia,pagado,A. Cruz,maria.santos@example,8888-1234,428,Ciudad abreviada
4,V-1004,"mayo 3, 2026",C004,Jose Rivera,La Ceiba,Monitor,Tecnología,0,3200,0,Tarjeta,Pendiente,L. Torres,jose.rivera@example.com,8877-6655,3200,Cantidad cero
5,V-1005,,C005,Sofia García,Choloma,Silla Oficina,Muebles,3,"1,250.00",10,efectivo,Cancelado,L. Torres,sofia.garcia@example.com,3366-2211,3375,Fecha faltante; descuento como entero
6,V-1006,2026-13-05,C006,Luis Martínez,Comayagua,Escritorio,Muebles,1,2500,0.15,Tarjeta,Pagado,M. Reyes,luis.martinez@example.com,9999-3344,2125,Fecha inválida
7,V-1007,2026-05-04,,Cliente Desconocido,Tegucigalpa,Impresora,Tecnología,1,3900,0,Transferencia,Pagado,A. Cruz,cliente.desconocido@example.com,9988-1111,3900,ClienteId faltante
8,V-1008,2026-05-04,C008,Karla Núñez,la ceiba,Papel,Oficina,10,85,0,Efectivo,Entregado,M. Reyes,karla.nunez@example.com,2233-4455,850,Estado fuera de catálogo
9,V-1009,2026-05-05,C009,Roberto Díaz,El Progreso,Tinta,Oficina,2,-250,0,Efectivo,Pagado,L. Torres,roberto.diaz@example.com,9988-2222,-500,Precio negativo


In [10]:
# ---------------------------------------------------------
# 3. Perfilado inicial
# ---------------------------------------------------------
# El perfilado nos ayuda a entender el estado de la data antes de limpiarla.

perfil_columnas = pd.DataFrame({
    "Columna": df_raw.columns,
    "TipoDetectado": [str(df_raw[col].dtype) for col in df_raw.columns],
    "Nulos_o_Vacios": [(df_raw[col].astype(str).str.strip() == "").sum() for col in df_raw.columns],
    "Valores_Unicos": [df_raw[col].nunique(dropna=False) for col in df_raw.columns]
})

display(perfil_columnas)

print("Duplicados exactos:", df_raw.duplicated().sum())
print("Duplicados por IdVenta:", df_raw["IdVenta"].duplicated().sum())


,Columna,TipoDetectado,Nulos_o_Vacios,Valores_Unicos
0,IdVenta,str,0,50
1,FechaVenta,object,1,31
2,ClienteId,str,1,51
3,ClienteNombre,str,1,49
4,Ciudad,str,1,13
5,Producto,str,0,15
6,Categoria,str,0,9
7,Cantidad,object,1,15
8,PrecioUnitario,object,0,23
9,Descuento,object,0,12


Duplicados exactos: 0
Duplicados por IdVenta: 2


In [11]:
# ---------------------------------------------------------
# 4. Funciones auxiliares de limpieza
# ---------------------------------------------------------

def is_missing(value):
    """Evalúa si un valor está vacío o nulo."""
    return pd.isna(value) or (isinstance(value, str) and value.strip() == "")

def clean_text(value):
    """Quita espacios extremos y colapsa dobles espacios internos."""
    if is_missing(value):
        return None
    return re.sub(r"\s+", " ", str(value).strip())

def key_text(value):
    """
    Genera una llave normalizada para comparar catálogos:
    - Sin espacios extremos
    - Minúsculas
    - Sin acentos
    """
    text = clean_text(value)
    if text is None:
        return None
    text = text.lower()
    text = "".join(
        char for char in unicodedata.normalize("NFD", text)
        if unicodedata.category(char) != "Mn"
    )
    return text

spanish_months = {
    "enero": "January", "febrero": "February", "marzo": "March", "abril": "April",
    "mayo": "May", "junio": "June", "julio": "July", "agosto": "August",
    "septiembre": "September", "setiembre": "September", "octubre": "October",
    "noviembre": "November", "diciembre": "December"
}

def parse_date(value):
    """
    Convierte fechas a date.
    No usamos inferencia libre para evitar aceptar fechas ambiguas o incorrectas.
    """
    if is_missing(value):
        return None, "Fecha faltante"

    if isinstance(value, (pd.Timestamp, datetime)):
        return pd.to_datetime(value).date(), None

    text = clean_text(value)
    text_lower = text.lower()

    for es, en in spanish_months.items():
        text_lower = text_lower.replace(es, en)

    text_normalized = text_lower.title()

    valid_formats = [
        "%Y-%m-%d",   # 2026-05-01
        "%Y/%m/%d",   # 2026/05/01
        "%d/%m/%Y",   # 13/05/2026
        "%Y.%m.%d",   # 2026.05.14
        "%m-%d-%Y",   # 05-06-2026
        "%B %d, %Y"   # May 3, 2026
    ]

    for fmt in valid_formats:
        try:
            return datetime.strptime(text_normalized, fmt).date(), None
        except ValueError:
            pass

    return None, f"Fecha inválida: {text}"

def parse_number(value):
    """
    Convierte números que vienen como:
    - 14500
    - L. 350.00
    - 1,250.00
    - 450,50
    - 1.250,00
    - 2 500
    - 14500L
    """
    if is_missing(value):
        return None, "Valor faltante"

    if isinstance(value, (int, float, np.integer, np.floating)) and not pd.isna(value):
        return float(value), None

    original = clean_text(value)
    text = original.lower()

    if text in {"n/a", "na", "nan", "none", "null"}:
        return None, f"Valor no numérico: {original}"

    text = text.replace("l.", "").replace("l", "")
    text = text.replace(" ", "")
    text = re.sub(r"[^0-9,\.\-]", "", text)

    if text in {"", "-", ".", ","}:
        return None, f"Valor no numérico: {original}"

    if "." in text and "," in text:
        if text.rfind(",") > text.rfind("."):
            # Formato europeo: 1.250,00
            text = text.replace(".", "").replace(",", ".")
        else:
            # Formato US: 1,250.00
            text = text.replace(",", "")
    elif "," in text:
        # Si la parte decimal tiene 2 dígitos, asumimos coma decimal.
        parts = text.split(",")
        if len(parts[-1]) == 2:
            text = text.replace(",", ".")
        else:
            text = text.replace(",", "")

    try:
        return float(text), None
    except ValueError:
        return None, f"Valor no numérico: {original}"

def parse_quantity(value):
    """
    Convierte cantidad a entero positivo.
    Acepta casos como '3 unidades' extrayendo el número inicial.
    """
    if is_missing(value):
        return None, "Cantidad faltante"

    if isinstance(value, (int, float, np.integer, np.floating)) and not pd.isna(value):
        quantity = float(value)
    else:
        text = clean_text(value).lower()
        match = re.match(r"^\s*(\d+)", text)
        if match:
            quantity = float(match.group(1))
        else:
            return None, f"Cantidad no numérica: {value}"

    if not quantity.is_integer():
        return None, f"Cantidad no entera: {value}"

    quantity = int(quantity)

    if quantity <= 0:
        return quantity, f"Cantidad inválida: {quantity}"

    return quantity, None

def parse_discount(value):
    """
    Normaliza descuentos:
    - 0.05 -> 0.05
    - 5% -> 0.05
    - "10" -> 0.10 porque viene como texto de porcentaje
    - N/A -> se asume 0 como advertencia
    """
    if is_missing(value):
        return 0.0, "Descuento faltante; se asumió 0"

    if isinstance(value, (int, float, np.integer, np.floating)) and not pd.isna(value):
        discount = float(value)
        if 0 <= discount <= 1:
            return discount, None
        return discount, f"Descuento fuera de rango: {discount}"

    text = clean_text(value)

    if text.lower() in {"n/a", "na", "nan", "none", "null"}:
        return 0.0, "Descuento no numérico; se asumió 0"

    if text.endswith("%"):
        number, error = parse_number(text[:-1])
        if error:
            return None, error
        discount = number / 100
    else:
        number, error = parse_number(text)
        if error:
            return None, error

        # Si viene como texto "10", lo interpretamos como 10%.
        discount = number / 100 if number > 1 else number

    if 0 <= discount <= 1:
        return discount, None

    return discount, f"Descuento fuera de rango: {text}"

def normalize_from_catalog(value, catalog_map, field_name):
    """Normaliza un valor contra un catálogo controlado."""
    key = key_text(value)

    if key is None:
        return None, f"{field_name} faltante"

    if key in catalog_map:
        return catalog_map[key], None

    return clean_text(value), f"{field_name} fuera de catálogo: {value}"

email_regex = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

def validate_email(value):
    """Valida email. En este ejercicio lo dejamos como advertencia, no como rechazo crítico."""
    if is_missing(value):
        return None, "Email faltante"

    email = clean_text(value).lower()

    if email_regex.match(email):
        return email, None

    return email, f"Email inválido: {value}"


In [12]:
# ---------------------------------------------------------
# 5. Catálogos válidos y equivalencias
# ---------------------------------------------------------

city_map = {
    "tegucigalpa": "Tegucigalpa",
    "tegus": "Tegucigalpa",
    "san pedro sula": "San Pedro Sula",
    "sps": "San Pedro Sula",
    "la ceiba": "La Ceiba",
    "choloma": "Choloma",
    "comayagua": "Comayagua",
    "el progreso": "El Progreso",
}

product_map = {
    "laptop": "Laptop",
    "mouse": "Mouse",
    "teclado": "Teclado",
    "monitor": "Monitor",
    "impresora": "Impresora",
    "silla oficina": "Silla Oficina",
    "escritorio": "Escritorio",
    "papel": "Papel",
    "tinta": "Tinta",
    "tablet": "Tablet",
    "audifonos": "Audífonos",
}

category_map = {
    "tecnologia": "Tecnología",
    "tech": "Tecnología",
    "muebles": "Muebles",
    "oficina": "Oficina",
    "accesorios": "Accesorios",
}

payment_map = {
    "efectivo": "Efectivo",
    "tarjeta": "Tarjeta",
    "transferencia": "Transferencia",
    "transferencia bancaria": "Transferencia",
}

status_map = {
    "pagado": "Pagado",
    "pendiente": "Pendiente",
    "cancelado": "Cancelado",
}


In [13]:
# ---------------------------------------------------------
# 6. Proceso de limpieza Raw → Silver + Rechazados
# ---------------------------------------------------------

def clean_sales_data(df):
    silver_rows = []
    rejected_rows = []
    log_rows = []

    seen_ids = set()
    exact_duplicate_mask = df.duplicated(keep="first")

    for index, row in df.iterrows():
        excel_row = index + 2  # +2 porque en Excel la fila 1 es el encabezado.
        errors = []
        warnings = []

        # -------------------------
        # Identificadores
        # -------------------------
        id_venta = clean_text(row.get("IdVenta"))

        if not id_venta:
            errors.append(("IdVenta faltante", "IdVenta", row.get("IdVenta")))
        elif id_venta in seen_ids:
            errors.append(("Duplicado de IdVenta", "IdVenta", id_venta))

        if exact_duplicate_mask.iloc[index]:
            errors.append(("Duplicado exacto", "IdVenta", id_venta))

        # -------------------------
        # Fecha
        # -------------------------
        fecha_venta, error = parse_date(row.get("FechaVenta"))
        if error:
            errors.append((error, "FechaVenta", row.get("FechaVenta")))

        # -------------------------
        # Cliente
        # -------------------------
        cliente_id = clean_text(row.get("ClienteId"))
        if cliente_id:
            cliente_id = cliente_id.upper()
        else:
            errors.append(("ClienteId faltante", "ClienteId", row.get("ClienteId")))

        cliente_nombre = clean_text(row.get("ClienteNombre"))
        if not cliente_nombre:
            errors.append(("ClienteNombre faltante", "ClienteNombre", row.get("ClienteNombre")))

        # -------------------------
        # Catálogos
        # -------------------------
        ciudad, error = normalize_from_catalog(row.get("Ciudad"), city_map, "Ciudad")
        if error:
            errors.append((error, "Ciudad", row.get("Ciudad")))

        producto, error = normalize_from_catalog(row.get("Producto"), product_map, "Producto")
        if error:
            errors.append((error, "Producto", row.get("Producto")))

        categoria, error = normalize_from_catalog(row.get("Categoria"), category_map, "Categoria")
        if error:
            errors.append((error, "Categoria", row.get("Categoria")))

        metodo_pago, error = normalize_from_catalog(row.get("MetodoPago"), payment_map, "MetodoPago")
        if error:
            errors.append((error, "MetodoPago", row.get("MetodoPago")))

        estado, error = normalize_from_catalog(row.get("Estado"), status_map, "Estado")
        if error:
            errors.append((error, "Estado", row.get("Estado")))

        # -------------------------
        # Números
        # -------------------------
        cantidad, error = parse_quantity(row.get("Cantidad"))
        if error:
            errors.append((error, "Cantidad", row.get("Cantidad")))
        elif isinstance(row.get("Cantidad"), str) and not str(row.get("Cantidad")).strip().isdigit():
            warnings.append("Cantidad con texto adicional; se extrajo número")

        precio_unitario, error = parse_number(row.get("PrecioUnitario"))
        if error:
            errors.append((error, "PrecioUnitario", row.get("PrecioUnitario")))
        elif precio_unitario <= 0:
            errors.append((f"Precio inválido: {precio_unitario}", "PrecioUnitario", row.get("PrecioUnitario")))

        descuento, error = parse_discount(row.get("Descuento"))
        if error:
            if "asumió 0" in error or "faltante" in error:
                warnings.append(error)
            else:
                errors.append((error, "Descuento", row.get("Descuento")))

        # -------------------------
        # Datos no críticos
        # -------------------------
        vendedor = clean_text(row.get("Vendedor"))
        email, email_warning = validate_email(row.get("EmailCliente"))
        telefono = clean_text(row.get("Telefono"))

        if email_warning:
            warnings.append(email_warning)

        total_reportado, error = parse_number(row.get("TotalReportado"))
        if error:
            warnings.append(f"TotalReportado no numérico o faltante: {row.get('TotalReportado')}")

        # -------------------------
        # Decisión final: Silver o Rechazados
        # -------------------------
        if errors:
            for reason, field, original_value in errors:
                rejected_rows.append({
                    "IdVenta": id_venta,
                    "FilaRaw": excel_row,
                    "MotivoRechazo": reason,
                    "CampoAfectado": field,
                    "ValorOriginal": "" if pd.isna(original_value) else original_value,
                    "AccionTomada": "Enviar a revisión / corregir en origen",
                    "FechaRevision": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "Revisor": "Proceso automático",
                    "Observaciones_Origen": row.get("Observaciones_Origen")
                })

            log_rows.append({
                "FechaHora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "Paso": "Validación",
                "Hoja/Archivo": "Ventas_Raw",
                "Acción realizada": f"Fila {excel_row} rechazada",
                "Resultado": "Rechazado",
                "Observaciones": "; ".join([error[0] for error in errors])
            })

        else:
            total_calculado = round(cantidad * precio_unitario * (1 - descuento), 2)
            diferencia = None if total_reportado is None else round(total_reportado - total_calculado, 2)

            if diferencia is not None and abs(diferencia) > 0.01:
                warnings.append(f"TotalReportado difiere del TotalCalculado por {diferencia}")

            silver_rows.append({
                "IdVenta": id_venta,
                "FechaVenta": fecha_venta.isoformat(),
                "ClienteId": cliente_id,
                "ClienteNombre": cliente_nombre,
                "Ciudad": ciudad,
                "Producto": producto,
                "Categoria": categoria,
                "Cantidad": cantidad,
                "PrecioUnitario": round(precio_unitario, 2),
                "Descuento": round(descuento, 4),
                "MetodoPago": metodo_pago,
                "Estado": estado,
                "Vendedor": vendedor,
                "EmailCliente": email,
                "EmailValido": "Sí" if not email_warning else "No",
                "Telefono": telefono,
                "TotalCalculado": total_calculado,
                "TotalReportado": total_reportado,
                "Diferencia": diferencia,
                "ObservacionesLimpieza": "; ".join(warnings) if warnings else "Registro limpio/normalizado"
            })

            seen_ids.add(id_venta)

            log_rows.append({
                "FechaHora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "Paso": "Limpieza",
                "Hoja/Archivo": "Ventas_Raw",
                "Acción realizada": f"Fila {excel_row} limpiada y enviada a Silver",
                "Resultado": "Correcto",
                "Observaciones": "; ".join(warnings) if warnings else "Sin observaciones"
            })

    return (
        pd.DataFrame(silver_rows),
        pd.DataFrame(rejected_rows),
        pd.DataFrame(log_rows)
    )

df_silver, df_rechazados, df_log = clean_sales_data(df_raw)

print("Registros enviados a Silver:", len(df_silver))
print("Registros rechazados:", df_rechazados["FilaRaw"].nunique())
display(df_silver.head())
display(df_rechazados.head())


Registros enviados a Silver: 34
Registros rechazados: 18


,IdVenta,FechaVenta,ClienteId,ClienteNombre,Ciudad,Producto,Categoria,Cantidad,PrecioUnitario,Descuento,MetodoPago,Estado,Vendedor,EmailCliente,EmailValido,Telefono,TotalCalculado,TotalReportado,Diferencia,ObservacionesLimpieza
0,V-1001,2026-05-01,C001,Ana López,Tegucigalpa,Laptop,Tecnología,1,14500.0,0.05,Tarjeta,Pagado,M. Reyes,ana.lopez@example.com,Sí,9988-7766,13775.00,13775.0,0.00,Registro limpio/normalizado
1,V-1002,2026-05-01,C002,Carlos Mejía,San Pedro Sula,Mouse,Tecnología,2,350.0,0.00,Efectivo,Pagado,A. Cruz,carlos.mejia@example.com,Sí,9999 1122,700.00,700.0,0.00,Registro limpio/normalizado
2,V-1003,2026-05-02,C003,María Santos,San Pedro Sula,Teclado,Tecnología,1,450.5,0.05,Transferencia,Pagado,A. Cruz,maria.santos@example,No,8888-1234,427.97,428.0,0.03,Email inválido: maria.santos@example; TotalRep...
3,V-1013,2026-05-07,C013,Mario Castro,La Ceiba,Silla Oficina,Muebles,1,1250.0,0.10,Efectivo,Pagado,A. Cruz,mario.castro@example.com,Sí,9988-4444,1125.00,1250.0,125.00,TotalReportado difiere del TotalCalculado por ...
4,V-1014,2026-05-08,C014,Paola Fernández,Choloma,Escritorio,Muebles,1,2500.0,0.05,Transferencia,Pagado,M. Reyes,paola.fernandez@example.com,Sí,9988-5555,2375.00,2375.0,0.00,Registro limpio/normalizado


,IdVenta,FilaRaw,MotivoRechazo,CampoAfectado,ValorOriginal,AccionTomada,FechaRevision,Revisor,Observaciones_Origen
0,V-1001,3,Duplicado de IdVenta,IdVenta,V-1001,Enviar a revisión / corregir en origen,2026-05-28 16:25:51,Proceso automático,Posible duplicado exacto
1,V-1004,6,Cantidad inválida: 0,Cantidad,0,Enviar a revisión / corregir en origen,2026-05-28 16:25:51,Proceso automático,Cantidad cero
2,V-1005,7,Fecha faltante,FechaVenta,,Enviar a revisión / corregir en origen,2026-05-28 16:25:51,Proceso automático,Fecha faltante; descuento como entero
3,V-1006,8,Fecha inválida: 2026-13-05,FechaVenta,2026-13-05,Enviar a revisión / corregir en origen,2026-05-28 16:25:51,Proceso automático,Fecha inválida
4,V-1007,9,ClienteId faltante,ClienteId,,Enviar a revisión / corregir en origen,2026-05-28 16:25:51,Proceso automático,ClienteId faltante


In [14]:
# ---------------------------------------------------------
# 7. Crear perfilado final y capa Gold
# ---------------------------------------------------------

df_perfilado = pd.DataFrame([
    {
        "Métrica": "Total filas raw",
        "Valor": len(df_raw),
        "Interpretación": "Registros recibidos en Ventas_Raw"
    },
    {
        "Métrica": "Total columnas raw",
        "Valor": df_raw.shape[1],
        "Interpretación": "Columnas recibidas en Ventas_Raw"
    },
    {
        "Métrica": "Registros válidos enviados a Silver",
        "Valor": len(df_silver),
        "Interpretación": "Filas que pasaron reglas críticas"
    },
    {
        "Métrica": "Filas raw rechazadas",
        "Valor": df_rechazados["FilaRaw"].nunique(),
        "Interpretación": "Filas con al menos un error crítico"
    },
    {
        "Métrica": "Motivos de rechazo diferentes",
        "Valor": df_rechazados["MotivoRechazo"].nunique(),
        "Interpretación": "Tipos de problemas detectados"
    },
    {
        "Métrica": "Total calculado Silver",
        "Valor": round(df_silver["TotalCalculado"].sum(), 2),
        "Interpretación": "Ventas calculadas usando registros limpios"
    },
    {
        "Métrica": "Registros con diferencia de total",
        "Valor": int((df_silver["Diferencia"].abs() > 0.01).sum()),
        "Interpretación": "Filas válidas donde TotalReportado no coincide con TotalCalculado"
    }
])

df_gold_ciudad = (
    df_silver
    .groupby("Ciudad", as_index=False)
    .agg(
        Ventas=("IdVenta", "count"),
        Unidades=("Cantidad", "sum"),
        TotalCalculado=("TotalCalculado", "sum")
    )
    .sort_values("TotalCalculado", ascending=False)
)

df_gold_categoria = (
    df_silver
    .groupby("Categoria", as_index=False)
    .agg(
        Ventas=("IdVenta", "count"),
        Unidades=("Cantidad", "sum"),
        TotalCalculado=("TotalCalculado", "sum")
    )
    .sort_values("TotalCalculado", ascending=False)
)

df_gold_estado = (
    df_silver
    .groupby("Estado", as_index=False)
    .agg(
        Ventas=("IdVenta", "count"),
        TotalCalculado=("TotalCalculado", "sum")
    )
    .sort_values("TotalCalculado", ascending=False)
)

df_resumen_rechazos = (
    df_rechazados
    .groupby("MotivoRechazo", as_index=False)
    .agg(Cantidad=("IdVenta", "count"))
    .sort_values("Cantidad", ascending=False)
)

display(df_perfilado)
display(df_gold_ciudad)
display(df_resumen_rechazos)


,Métrica,Valor,Interpretación
0,Total filas raw,52.00,Registros recibidos en Ventas_Raw
1,Total columnas raw,17.00,Columnas recibidas en Ventas_Raw
2,Registros válidos enviados a Silver,34.00,Filas que pasaron reglas críticas
3,Filas raw rechazadas,18.00,Filas con al menos un error crítico
4,Motivos de rechazo diferentes,17.00,Tipos de problemas detectados
5,Total calculado Silver,128757.97,Ventas calculadas usando registros limpios
6,Registros con diferencia de total,4.00,Filas válidas donde TotalReportado no coincide...


,Ciudad,Ventas,Unidades,TotalCalculado
5,Tegucigalpa,9,9,42835.00
1,Comayagua,5,6,26802.50
4,San Pedro Sula,8,30,26585.47
2,El Progreso,1,1,14500.00
3,La Ceiba,7,57,12875.00
0,Choloma,4,8,5160.00


,MotivoRechazo,Cantidad
8,Duplicado de IdVenta,2
0,Cantidad faltante,1
1,Cantidad inválida: -1,1
3,Cantidad no numérica: dos,1
2,Cantidad inválida: 0,1
4,Ciudad faltante,1
5,ClienteId faltante,1
6,ClienteNombre faltante,1
7,Descuento fuera de rango: 1.2,1
9,Estado faltante,1
